In [ ]:
import pandas as pd

print(pd.__version__)


In [ ]:
cpi = pd.read_csv("C:/Users/novin/Documents/Project/Inflation_analysis/cpi.csv")
rate = pd.read_csv("C:/Users/novin/Documents/Project/Inflation_analysis/interest_rate.csv")

In [ ]:
p=cpi.head()

In [ ]:
print(p)

In [ ]:
cpi.rename(columns={"observation_date": "DATE"}, inplace=True)
rate.rename(columns={"observation_date": "DATE"}, inplace=True)



In [ ]:
print(cpi.head())
print(rate.head())


In [ ]:

cpi["DATE"] = pd.to_datetime(cpi["DATE"])
rate["DATE"] = pd.to_datetime(rate["DATE"])

In [ ]:
cpi = cpi.set_index("DATE")
rate = rate.set_index("DATE")


In [ ]:
cpi["inflation"] = cpi["CPIAUCSL"].pct_change(12) * 100


import  matplotlib
import  numpy as np
plt.figure(figsize=(12,5))

cpi["inflation"].plot()

plt.title("US Inflation Rate (YoY)")
plt.ylabel("Percent")
plt.xlabel("Year")

plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(12,5))

cpi["inflation"].plot()

plt.title("Inflation Rate")
plt.xlabel("Date")
plt.ylabel("Percent")

plt.show()


In [ ]:
data = cpi.join(rate)
data.head()


In [ ]:
plt.figure(figsize=(12,5))

data["inflation"].plot(label="Inflation")
data["FEDFUNDS"].plot(label="Interest Rate")

plt.legend()
plt.title("Inflation vs Interest Rate")

plt.show()


In [ ]:
mahasebeh=data[["inflation","FEDFUNDS"]].corr()


In [ ]:
print(mahasebeh)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Merge CPI and FEDFUNDS into one dataframe (if not already merged)
# Make sure both have 'inflation' and 'FEDFUNDS'
df = data.dropna(subset=["inflation", "FEDFUNDS"])

plt.figure(figsize=(10,6))
sns.regplot(
    x=df["inflation"],
    y=df["FEDFUNDS"],
    scatter_kws={"alpha": 0.5, "color": "orange"},
    line_kws={"color": "blue"}
)

plt.title("Relationship Between Inflation  and Federal Funds Rate")
plt.xlabel("Inflation Rate ")
plt.ylabel("Federal Funds Rate (%)")
plt.grid(True, linestyle="--", alpha=0.3)

plt.show()




In [ ]:

# Federal Reserve Response to Inflation Dynamics

### Analyzing the Time Lag Between Inflation Increases and Federal Reserve Interest Rate Decisions
import pandas as pd
import matplotlib.pyplot as plt

# Create lagged inflation variables
lags = range(0, 13)

correlations = []

for lag in lags:

    corr = data["inflation"].shift(lag).corr(data["FEDFUNDS"])

    correlations.append(corr)

# Plot
plt.figure(figsize=(10,6))

plt.plot(lags, correlations, marker="o", linewidth=2)

plt.title("Lag Correlation: Inflation vs Federal Funds Rate")
plt.xlabel("Lag (Months)")
plt.ylabel("Correlation")

plt.grid(True, linestyle="--", alpha=0.3)

plt.show()
#The lag correlation results show that the strongest relationship between
#inflation and the Federal Funds Rate occurs at lag 0 and lag 1. This indicates
#that the Federal Reserve reacts very quickly to changes in inflation, typically
#within the same month or the following month. At longer lags, the correlation
#weakens substantially, suggesting that older inflation data has little influence
#on current monetary policy decisions.



<h1 style="color:darkblue;">
Federal Reserve Response to Inflation
</h1>

<h3 style="color:darkred;">
How Many Months After Rising Inflation Does the Federal Reserve React?
</h3>


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

corr = df[['CPIAUCSL', 'inflation', 'FEDFUNDS']].corr()

plt.figure(figsize=(8,6))

sns.heatmap(
    corr,
    annot=True,
    cmap='coolwarm',
    fmt='.2f'
)

plt.title('Correlation Heatmap')
plt.show()
#Interest rates respond primarily to inflation rather than to the absolute level of prices.
#Consequently, the correlation between inflation and the Federal Funds Rate is strong and positive,
#whereas the relationship between CPI levels and interest rates is weaker and less stable.


In [ ]:
window = 12  # 12 months rolling window

rolling_corr = df['inflation'].rolling(window).corr(df['FEDFUNDS'])

plt.figure(figsize=(12,6))
plt.plot(df.index, rolling_corr, color='purple')
plt.axhline(0, color='black', linewidth=1)
plt.title(f'Rolling Correlation (window={window} months)')
plt.xlabel('Year')
plt.ylabel('Correlation')
plt.show()


In [ ]:
#forcast inflation with ARIMA
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller


In [ ]:
result = adfuller(data['inflation'].dropna())

print('ADF Statistic:', result[0])
print('p-value:', result[1])


In [ ]:
model = ARIMA(data['inflation'], order=(1,0,1))
model_fit = model.fit()


print(model_fit.summary())



In [ ]:

fitted_values = model_fit.fittedvalues


import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))


plt.plot(data["inflation"], label='Actual Inflation')

plt.plot(fitted_values, label='Fitted Inflation', linestyle='--')

plt.title('Actual vs Fitted Inflation (ARIMA Model)')
plt.legend()

plt.show()


In [ ]:

forecast = model_fit.get_forecast(steps=12)
forecast_df = forecast.summary_frame()

plt.figure(figsize=(10, 5))
plt.plot(data['inflation'].tail(50), label='Last 50 months') # نمایش ۵۰ ماه آخر برای وضوح بیشتر
plt.plot(forecast_df['mean'], label='Forecast', color='red')
plt.fill_between(forecast_df.index, forecast_df['mean_ci_lower'], forecast_df['mean_ci_upper'], color='pink', alpha=0.3)

plt.title('Inflation Forecast for Next 12 Months')
plt.legend()
plt.show()


In [ ]:

import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from statsmodels.tsa.arima.model import ARIMA


data = data.dropna(subset=['inflation'])

train_size = len(data) - 5
train = data['inflation'].iloc[:train_size]
test = data['inflation'].iloc[train_size:]


history = [x for x in train]
predictions = []


for t in range(len(test)):
    model = ARIMA(history, order=(1, 0, 1))
    model_fit = model.fit()
    output = model_fit.forecast()
    yhat = output[0]
    predictions.append(yhat)

    obs = test.iloc[t]
    history.append(obs)
    print(f'Month {t+1}: predicted={yhat:.2f}, actual={obs:.2f}')

# RMSE calculation
rmse = np.sqrt(mean_squared_error(test, predictions))
print(f'\n========================================')
print(f'Total RMSE: {rmse:.3f}')
print(f'========================================')



In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import het_arch

# 1. Fit ARIMA model on the full dataset (to extract Residuals)
# Using the same order (1,0,1) as before
model = ARIMA(data['inflation'], order=(1, 0, 1))
model_fit = model.fit()

# 2. Extract residuals
residuals = model_fit.resid

# 3. Statistical test for ARCH effects
# Engle's ARCH test checks whether the model's error variance is random or follows a pattern
test_stat, p_value, f_stat, f_p_value = het_arch(residuals)

print(f'ARCH test result:')
print(f'P-value: {p_value:.4f}')

if p_value < 0.05:
    print("\nResult: P-value is less than 0.05.")
    print("Your data shows ARCH effects! Congratulations.")
    print("A GARCH model could meaningfully improve your forecast accuracy.")
else:
    print("\nResult: P-value is greater than 0.05.")
    print("No significant ARCH effects were found; ARIMA is sufficient and GARCH would not add meaningful improvement.")

# 4. Plot the residuals
plt.figure(figsize=(10, 4))
plt.plot(residuals)
plt.title('Residuals of ARIMA Model')
plt.show()
# P-value for ARCH effect in the data is 0.0000, meaning it is highly significant.



In [ ]:

from arch import arch_model


garch_model = arch_model(residuals, vol='Garch', p=1, q=1)
garch_result = garch_model.fit(disp='off')


print(garch_result.summary())
garch_result.plot()
plt.show()


In [ ]:


model = arch_model(data['inflation'].dropna(),
                   mean='AR',  
                   lags=1,     
                   vol='Garch',
                   p=1, q=1)  


results = model.fit(disp='off')


print(results.summary())


forecasts = results.forecast(horizon=12)

print(forecasts.mean.iloc[-1])
print(forecasts.variance.iloc[-1])


In [ ]:
import matplotlib.pyplot as plt


forecasts = results.forecast(horizon=12)
mean_forecast = forecasts.mean.iloc[-1]

std_dev = forecasts.variance.iloc[-1]**0.5

plt.figure(figsize=(10, 5))
plt.plot(mean_forecast, label='Forecasted Inflation', color='red')

plt.fill_between(mean_forecast.index,
                 mean_forecast - 1.96 * std_dev,
                 mean_forecast + 1.96 * std_dev,
                 color='pink', alpha=0.3, label='95% Confidence Interval')

plt.title('ARIMA-GARCH Inflation Forecast')
plt.legend()
plt.show()
